In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:openai/gpt-oss-20b")
response=model.invoke("why do parrots talk ?")
response

AIMessage(content='**Short answer:**  \nParrots mimic human speech because they’re naturally *vocal learners* that use sound to communicate, bond, and entertain themselves. In the wild they use a wide range of calls to keep flocks together, warn of danger, and attract mates. In captivity, the “talk” is simply the next best thing to the complex social world they’re missing.  \n\n---\n\n## 1.  Why parrots can mimic at all\n\n| Feature | What it does | Why it matters for “talking” |\n|---------|--------------|------------------------------|\n| **Syrinx (bird larynx)** | Highly flexible, can produce a broad range of sounds | Allows precise control of pitch, tone, and rhythm |\n| **Large brain‑to‑body ratio** | Especially the *telencephalon* and *song nuclei* | Supports complex motor control and auditory memory |\n| **Vocal learning** | Ability to record sounds and reproduce them later | The neural circuitry (e.g., the *high vocal center* – HVC) is tuned for imitation |\n| **Social structur

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [7]:
response=model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f" Tool: {tool_call['name']}")
    print(f" Args:{tool_call['args']}")



content='' additional_kwargs={'reasoning_content': 'We need to call the function get_weather with location "Boston".', 'tool_calls': [{'id': 'fc_44b88f08-871e-4826-9432-2ce148a24dd8', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 127, 'total_tokens': 164, 'completion_time': 0.039515995, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.006388678, 'prompt_tokens_details': None, 'queue_time': 0.333397581, 'total_time': 0.045904673}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_84bb35977d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a08b02-2925-7552-bae0-b536d91fc8eb-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_44b88f08-871e-4826-9432-2ce148a24dd8', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_

### Tool Execution Loops


In [5]:
# Step 1: Model generates tool calls
messages = [{"role":"user", "content":"What is the weather in Boston"}]
ai_message=model_with_tools.invoke(messages)
messages.append(ai_message)

#Step 2: Execute tools and collect results

for tool_call in ai_message.tool_calls:
    #Execute the tool with the generated arguments
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)
# Step 3: Pass results back to model for final response
final_response =model_with_tools.invoke(messages)
print(final_response.text)


It’s sunny in Boston today. Enjoy the nice weather!


In [6]:
messages

[{'role': 'user', 'content': 'What is the weather in Boston'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call get_weather function with location "Boston".', 'tool_calls': [{'id': 'fc_d5dd829e-d35d-4427-8eb1-a40c1485b241', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 126, 'total_tokens': 162, 'completion_time': 0.048331263, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.006905893, 'prompt_tokens_details': None, 'queue_time': 0.345910507, 'total_time': 0.055237156}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_feb9b278f1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a08b00-246a-7a81-90dc-5e3032306432-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_d5dd829e-d35d-4427-8eb1-a40c1485